## Image Studio: A tool for image generation, background generation, and image manipulation.

Author: [Bhushan Garware](http://who/bgarware)

##### This software is designed for prototyping (Not for production use) and provided 'as-is', without any express or implied warranty.

In [1]:
import os
import sys
import io
import uuid
import functools
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import vertexai
from vertexai.preview.vision_models import ImageGenerationModel, Image as VertexImage
import gradio as gr
from rembg import remove
from PIL import Image, ImageFont, ImageDraw
import numpy as np

# Load environment variables if python-dotenv is available
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Ensure local temporary and font directories exist
TEMP_DIR = Path("./tmp")
FONTS_DIR = Path("./fonts")
TEMP_DIR.mkdir(parents=True, exist_ok=True)
FONTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Configure Vertex AI Project & Location
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("PROJECT_ID") or "gdc-ai-playground"
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION") or os.environ.get("LOCATION") or "us-central1"

try:
    import google.auth
    os.environ.setdefault("GOOGLE_CLOUD_QUOTA_PROJECT", PROJECT_ID)
    os.environ.setdefault("CLOUDSDK_CORE_PROJECT", PROJECT_ID)
    try:
        credentials, auth_project = google.auth.default(quota_project_id=PROJECT_ID)
        vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)
    except Exception:
        vertexai.init(project=PROJECT_ID, location=LOCATION)
    print(f"Vertex AI initialized successfully. Project: {PROJECT_ID}, Location: {LOCATION}")
except Exception as e:
    print(f"Notice: Vertex AI initialization encountered an issue: {e}")
    print("If running locally, set GOOGLE_CLOUD_PROJECT in your environment or .env file.")


In [3]:
# Function to generate images from the input prompt 
def image_generation_completion(input):
    """
    Generates 4 images from the input prompt using Imagen 4 (imagen-4.0-generate-001)
    and Gemini Native Image models (gemini-2.5-flash-image, gemini-3.1-flash-image, gemini-3-pro-image-preview).

    Args:
      input: The input prompt as a string.

    Returns:
      A list of four generated images, each represented as a PIL Image.
    """
    if not input or not str(input).strip():
        gr.Warning("Please enter an image prompt first.")
        return [None, None, None, None]

    prompt_text = str(input).strip()
    errors = []

    # 1. Try Imagen 4 and vision models
    for model_name in ["imagen-4.0-generate-001", "imagen-3.0-generate-002", "imagen-3.0-generate-001", "imagen-3.0-fast-generate-001"]:
        try:
            from vertexai.preview.vision_models import ImageGenerationModel
            model = ImageGenerationModel.from_pretrained(model_name)
            response = model.generate_images(
                prompt=prompt_text,
                number_of_images=4,
            )
            image_return_list = [img._pil_image for img in response.images if hasattr(img, '_pil_image')]
            if image_return_list:
                while len(image_return_list) < 4:
                    image_return_list.append(None)
                return image_return_list
        except Exception as e:
            errors.append(f"{model_name}: {e}")

    # 2. Try Gemini Native Image models (gemini-2.5-flash-image generates 4 in parallel)
    for model_name in ["gemini-2.5-flash-image", "gemini-3.1-flash-image", "gemini-3-pro-image-preview"]:
        try:
            from vertexai.generative_models import GenerativeModel
            from concurrent.futures import ThreadPoolExecutor
            model = GenerativeModel(model_name)

            def fetch_single_image(idx):
                try:
                    res = model.generate_content(
                        prompt_text,
                        generation_config={"response_modalities": ["TEXT", "IMAGE"]}
                    )
                    if res and res.candidates:
                        import io
                        for cand in res.candidates:
                            for p in cand.content.parts:
                                if hasattr(p, 'inline_data') and p.inline_data:
                                    return Image.open(io.BytesIO(p.inline_data.data))
                except Exception as ex:
                    print(f"[ImageStudio {model_name} #{idx} Error] {ex}")
                return None

            with ThreadPoolExecutor(max_workers=4) as executor:
                futures = [executor.submit(fetch_single_image, i) for i in range(4)]
                image_return_list = [f.result() for f in futures]
                image_return_list = [img for img in image_return_list if img is not None]

            if image_return_list:
                while len(image_return_list) < 4:
                    image_return_list.append(None)
                return image_return_list
        except Exception as e:
            errors.append(f"{model_name}: {e}")

    # If all failed, format and display the detailed error
    last_err = errors[-1] if errors else "Unknown error"
    error_msg = f"Image generation failed across all candidate models. Last error: {last_err}"
    print(f"[ImageStudio Error] {error_msg}")
    for err in errors:
        print(f"  [Model Attempt] {err}")
    gr.Warning(f"{error_msg}. Check your project billing/quota (IPM > 0) in GCP Console.")
    return [None, None, None, None]

In [4]:
def prompt_generation(persona, signal, theme, lighting, quality, extra_desc, TextOnImg, TextFmtOnImg):
    """
    Generates an enriched image generation prompt using Gemini / Vertex AI text models,
    with a robust fallback to structured template composition.
    """
    params_list = [p for p in [persona, signal, theme, lighting, quality, extra_desc] if p and str(p).strip()]
    params_list_str = ", ".join(params_list) if params_list else "A high quality product showcase"
    
    few_shot_prompt = f"""You are an expert in writing prompts for Image Generation Models. Using the provided phrases and keywords, concatenate them and add on some realistic details to generate a logical and meaningful prompt that can be used for image generation.

input: Young woman, wearing NIKE sneakers, tennis court, Natural, HD image, Photo.
output: A Photo of Young woman wearing NIKE sneakers on tennis court, Natural lighting, HD quality photo clicked by a professional photographer.
input: Old man, wearing sports shoes, vegetable market, warm, high-quality, sketch.
output: A sketch of old man wearing sports shoes in vegetable market, warm lighting, high quality sketch drawn by a professional painter.
input: {params_list_str}
output:"""

    output_prompt = ""
    # Try active Gemini text models on Vertex AI
    for model_name in ["gemini-2.5-flash-image", "gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro"]:
        try:
            from vertexai.generative_models import GenerativeModel
            model = GenerativeModel(model_name)
            response = model.generate_content(few_shot_prompt)
            if response and response.text:
                output_prompt = response.text.strip()
                break
        except Exception:
            continue

    # Fallback to local deterministic template composition if LLM is unavailable
    if not output_prompt:
        subject_part = persona or "subject"
        action_part = f", {signal}" if signal else ""
        theme_part = f" in {theme}" if theme else ""
        light_part = f", {lighting} lighting" if lighting else ""
        qual_part = f", {quality}" if quality else ""
        type_part = f"A {extra_desc} of" if extra_desc else "A photo of"
        output_prompt = f"{type_part} {subject_part}{action_part}{theme_part}{light_part}{qual_part}."

    # Add text overlay instructions if specified
    if TextOnImg and str(TextOnImg).strip():
        txt = str(TextOnImg).strip()
        if TextFmtOnImg and str(TextFmtOnImg).strip():
            output_prompt += f" Add a title to the corner that reads '{txt}' in {str(TextFmtOnImg).strip()}."
        else:
            output_prompt += f" Add a title to the corner that reads '{txt}'."
            
    return output_prompt

In [5]:
def background_generation(input_image, prompt):
    """
    Generates three background variations for a given input image based on a text prompt.
    Supports Vertex AI Imagen capability models with seamless fallback to Gemini Multimodal image editing.
    """
    if input_image is None:
        gr.Warning("Please upload an input image.")
        return [None, None, None]
        
    if not prompt or not str(prompt).strip():
        gr.Warning("Please provide a background prompt.")
        return [None, None, None]

    prompt_text = str(prompt).strip()
    req_id = uuid.uuid4().hex[:8]
    base_img_path = TEMP_DIR / f"base_img_{req_id}.png"
    created_temp_files = [base_img_path]

    try:
        input_image.save(str(base_img_path))
        
        # 1. Try Imagen editing models
        try:
            from vertexai.preview.vision_models import Image as VertexImage, ImageGenerationModel
            model = None
            for model_name in ["imagen-4.0-generate-001", "imagen-3.0-capability-001", "imagen-3.0-generate-002"]:
                try:
                    model = ImageGenerationModel.from_pretrained(model_name)
                    break
                except Exception:
                    continue

            if model is not None:
                base_img = VertexImage.load_from_file(location=str(base_img_path))
                images = model.edit_image(
                    base_image=base_img,
                    prompt=prompt_text,
                    number_of_images=3,
                    edit_mode="product-image"
                )

                results = []
                for i, img in enumerate(images):
                    save_path = TEMP_DIR / f"im_{req_id}_{i}.png"
                    created_temp_files.append(save_path)
                    img.save(location=str(save_path), include_generation_parameters=True)
                    opened = Image.open(str(save_path)).resize((1024, 1024), Image.LANCZOS)
                    results.append(opened)

                if results:
                    while len(results) < 3:
                        results.append(None)
                    return results
        except Exception as imagen_err:
            print(f"[ImageStudio Notice] Imagen background editing unavailable ({imagen_err}), switching to Gemini multimodal...")

        # 2. Fallback: Gemini Native Multimodal background synthesis (3 in parallel)
        for model_name in ["gemini-2.5-flash-image", "gemini-3.1-flash-image", "gemini-3-pro-image-preview"]:
            try:
                from vertexai.generative_models import GenerativeModel, Part
                model = GenerativeModel(model_name)

                # Convert input PIL image to bytes part for multimodal query
                img_byte_arr = io.BytesIO()
                input_image.save(img_byte_arr, format='PNG')
                img_part = Part.from_data(data=img_byte_arr.getvalue(), mime_type="image/png")
                edit_instruction = f"Place this product into a new background scene: {prompt_text}. Maintain product fidelity and lighting coherence."

                def fetch_bg_variation(idx):
                    try:
                        res = model.generate_content(
                            [img_part, edit_instruction],
                            generation_config={"response_modalities": ["TEXT", "IMAGE"]}
                        )
                        if res and res.candidates:
                            for cand in res.candidates:
                                for p in cand.content.parts:
                                    if hasattr(p, 'inline_data') and p.inline_data:
                                        return Image.open(io.BytesIO(p.inline_data.data)).resize((1024, 1024), Image.LANCZOS)
                    except Exception as ex:
                        print(f"[ImageStudio Background {model_name} #{idx} Error] {ex}")
                    return None

                with ThreadPoolExecutor(max_workers=3) as executor:
                    futures = [executor.submit(fetch_bg_variation, i) for i in range(3)]
                    results = [f.result() for f in futures]
                    results = [img for img in results if img is not None]

                if results:
                    while len(results) < 3:
                        results.append(None)
                    return results
            except Exception as gemini_err:
                print(f"[ImageStudio Error] Gemini background variation failed on {model_name}: {gemini_err}")

        gr.Warning("Background generation failed across candidate models. Verify Vertex AI project & billing.")
        return [None, None, None]
    except Exception as e:
        error_msg = f"Background generation failed: {e}"
        print(f"[ImageStudio Error] {error_msg}")
        gr.Warning(f"{error_msg}. Verify Vertex AI permissions.")
        return [None, None, None]
    finally:
        # Clean up temporary disk buffers
        for temp_file in created_temp_files:
            try:
                if temp_file.exists():
                    temp_file.unlink()
            except Exception:
                pass


def insert_image(im1, im2, angle, height, width, left, top):
    """
    Inserts a product/foreground image (im2) into a base background image (im1)
    after resizing, rotating, and cleanly removing the background using rembg.
    """
    if im1 is None:
        gr.Warning("Please upload a Background Image.")
        return None
    if im2 is None:
        gr.Warning("Please upload a Product Image to insert.")
        return im1
        
    try:
        base_img = im1.copy().convert("RGBA")
        
        # Remove background of product image to get RGBA with transparent alpha
        fg_rgba = remove(im2)
        if fg_rgba.mode != "RGBA":
            fg_rgba = fg_rgba.convert("RGBA")
        
        # Resize foreground image
        target_w = max(10, int(width))
        target_h = max(10, int(height))
        fg_resized = fg_rgba.resize((target_w, target_h), Image.LANCZOS)
        
        # Rotate foreground image (counter-clockwise)
        fg_rotated = fg_resized.rotate(-float(angle), resample=Image.BICUBIC, expand=True)
        
        # Create transparent overlay layer matching base image size
        overlay_layer = Image.new("RGBA", base_img.size, (0, 0, 0, 0))
        overlay_layer.paste(fg_rotated, (int(left), int(top)), mask=fg_rotated)
        
        # Alpha composite base and overlay
        result = Image.alpha_composite(base_img, overlay_layer)
        return result.convert("RGB")
    except Exception as e:
        gr.Warning(f"Error inserting image: {e}")
        return im1


def insert_more_images(input_image):
    """
    Allows chaining insertions: moves the output image back into the background slot.
    """
    return [input_image, None, None]


def AddLogo(MainImage, LogoImage, factor, opacity, left, top):
    """
    Adds a logo overlay onto a main image with scaling, opacity, and positioning.
    """
    if MainImage is None:
        gr.Warning("Please upload a Background Image.")
        return None
    if LogoImage is None:
        gr.Warning("Please upload a Logo Image.")
        return MainImage

    try:
        # Load main image
        if isinstance(MainImage, str):
            main_pil = Image.open(MainImage).convert("RGBA")
        elif isinstance(MainImage, np.ndarray):
            main_pil = Image.fromarray(MainImage).convert("RGBA")
        else:
            main_pil = MainImage.copy().convert("RGBA")

        # Load logo image
        if isinstance(LogoImage, str):
            logo_pil = Image.open(LogoImage).convert("RGBA")
        elif isinstance(LogoImage, np.ndarray):
            logo_pil = Image.fromarray(LogoImage).convert("RGBA")
        else:
            logo_pil = LogoImage.copy().convert("RGBA")

        # Resize logo by factor
        orig_w, orig_h = logo_pil.size
        new_w = max(5, int(orig_w * float(factor)))
        new_h = max(5, int(orig_h * float(factor)))
        logo_resized = logo_pil.resize((new_w, new_h), Image.LANCZOS)

        # Adjust opacity
        alpha_scale = max(0.0, min(1.0, float(opacity) / 100.0))
        r, g, b, a = logo_resized.split()
        a = a.point(lambda p: int(p * alpha_scale))
        logo_resized = Image.merge("RGBA", (r, g, b, a))

        # Composite onto canvas
        overlay_layer = Image.new("RGBA", main_pil.size, (0, 0, 0, 0))
        overlay_layer.paste(logo_resized, (int(left), int(top)), mask=logo_resized)
        
        result = Image.alpha_composite(main_pil, overlay_layer)
        return result.convert("RGB")
    except Exception as e:
        gr.Warning(f"Error adding logo: {e}")
        return MainImage


@functools.lru_cache(maxsize=128)
def find_font(font_name, font_size):
    """
    Resolves font by checking fonts directory, temp directory, system font directories,
    and falls back safely to default font to avoid OSError.
    """
    font_size = max(8, int(font_size))
    candidate_paths = [
        FONTS_DIR / f"{font_name}.ttf",
        FONTS_DIR / f"{font_name}.otf",
        TEMP_DIR / f"{font_name}.ttf",
        TEMP_DIR / f"{font_name}.otf",
        Path(f"/System/Library/Fonts/Supplemental/{font_name}.ttf"),
        Path(f"/System/Library/Fonts/{font_name}.ttf"),
        Path(f"/Library/Fonts/{font_name}.ttf"),
        Path(f"/usr/share/fonts/truetype/{font_name}.ttf"),
        Path(f"C:/Windows/Fonts/{font_name}.ttf"),
    ]
    for p in candidate_paths:
        if p.exists():
            try:
                return ImageFont.truetype(str(p), font_size)
            except Exception:
                pass

    try:
        return ImageFont.truetype(font_name, font_size)
    except Exception:
        pass

    try:
        return ImageFont.load_default(size=font_size)
    except TypeError:
        return ImageFont.load_default()


def AddText(bkg_image, input_text, font, font_size, R, G, B, left, top):
    """
    Renders custom styled text onto an image.
    """
    if bkg_image is None:
        gr.Warning("Please upload a Background Image.")
        return None
    if not input_text:
        return bkg_image

    try:
        img = bkg_image.copy().convert("RGB")
        draw = ImageDraw.Draw(img)
        loaded_font = find_font(font, int(font_size))
        color = (int(R), int(G), int(B))
        draw.text((int(left), int(top)), str(input_text), fill=color, font=loaded_font)
        return img
    except Exception as e:
        gr.Warning(f"Error adding text: {e}")
        return bkg_image


def AddMoreText(input_image):
    """
    Allows chaining text additions: moves the output image back into the background slot.
    """
    return [input_image, input_image, ""]

In [6]:
available_fonts = ["Arial", "Arial Black", "Arial Bold", "Arial Italic", "Arial Narrow", "Arial Rounded Bold"]
if FONTS_DIR.exists():
    for f in FONTS_DIR.glob("*.ttf"):
        if f.stem not in available_fonts:
            available_fonts.append(f.stem)
for extra in ["ClarendonBT", "FUTURAM", "SerpentineBoldItalic", "Helvetica", "Courier New", "Times New Roman"]:
    if extra not in available_fonts:
        available_fonts.append(extra)

gr.close_all()
with gr.Blocks(theme=gr.themes.Soft(), title="Image Studio") as demo:
    with gr.Tab("Image Generation"):
        # Prompt Generation Part
        with gr.Row():
            with gr.Column(scale=1):
                Persona = gr.Textbox(label="Subject", info="e.g. Old woman, Man in 60s, Winter Boots")
            with gr.Column(scale=1):
                Signals = gr.Textbox(label="Action", info="e.g. Standing, kept on the table")
            with gr.Column(scale=1):
                Theme = gr.Textbox(label="Theme", info="e.g. On tennis court, in the market")
        with gr.Row():
            with gr.Column(scale=1):
                photo_modifiers = gr.Dropdown(["Dramatic", "Natural", "Warm", "Cold", "Cinematic"], label="Photography Modifiers", value="Natural")
            with gr.Column(scale=1):
                quality_modifiers = gr.Dropdown(["high-quality", "beautiful", "stylized", "4K", "HDR", "By a professional photographer"], label="Image Quality Modifier", value="By a professional photographer")
            with gr.Column(scale=1):
                other_desc = gr.Dropdown(["Photo", "painting", "Sketch", "Digital Art"], label="Image Type", value="Photo")

        with gr.Row():
            with gr.Column(scale=1):
                TextOnImg = gr.Textbox(label="Optional, Text on Image", info="e.g. 30% OFF on Flight Bookings")
            with gr.Column(scale=1):
                TextFmtOnImg = gr.Textbox(label="Optional, Text format", info="e.g. Pink and white block letters")      
            
        with gr.Row():
            btn = gr.Button("Generate Prompt", variant="secondary")    
        
        # Image Generation part
        with gr.Row():
            with gr.Column(scale=1):
                image_prompt = gr.Textbox(label="Image Generation Prompt", lines=3)
                
        btn.click(fn=prompt_generation, inputs=[Persona, Signals, Theme, photo_modifiers, quality_modifiers, other_desc, TextOnImg, TextFmtOnImg], outputs=image_prompt)

        with gr.Row():
            with gr.Column(scale=1):    
                img_btn = gr.Button("Generate Images", variant="primary")

        with gr.Row():
            with gr.Column():
                output_image_1 = gr.Image(label="Result Image 1", type="pil")
            with gr.Column():
                output_image_2 = gr.Image(label="Result Image 2", type="pil")
        with gr.Row():
            with gr.Column():
                output_image_3 = gr.Image(label="Result Image 3", type="pil")
            with gr.Column():
                output_image_4 = gr.Image(label="Result Image 4", type="pil")

        components = [image_prompt, output_image_1, output_image_2, output_image_3, output_image_4]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")

        img_btn.click(fn=image_generation_completion, inputs=[image_prompt], outputs=[output_image_1, output_image_2, output_image_3, output_image_4])
        
    with gr.Tab("Background Generation"):
        with gr.Row():
            with gr.Column():
                b_input_image = gr.Image(label="Input Image", type="pil")
            with gr.Column():
                b_text_prompt = gr.Textbox(label="Background Prompt", info="e.g. Blue sea with white sand", lines=3)
                b_btn = gr.Button("Change Background", variant="primary")
        with gr.Row():
            with gr.Column():
                b_output_image1 = gr.Image(label="Result Image 1", type="pil")
            with gr.Column():
                b_output_image2 = gr.Image(label="Result Image 2", type="pil")
            with gr.Column():
                b_output_image3 = gr.Image(label="Result Image 3", type="pil")
                    
        b_components = [b_input_image, b_text_prompt, b_output_image1, b_output_image2, b_output_image3]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(b_components)
                    
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        b_btn.click(fn=background_generation, inputs=[b_input_image, b_text_prompt], outputs=[b_output_image1, b_output_image2, b_output_image3])
    
    with gr.Tab("Insert Image"):
        with gr.Row():
            with gr.Column():
                i_bkg_image = gr.Image(label="Background Image", type="pil")
            with gr.Column():
                i_prd_image = gr.Image(label="Product Image", type="pil")
        with gr.Row():
            with gr.Column():
                i_angle = gr.Slider(-180, 180, value=0, label="Angle", info="Angle of Rotation")
                i_height = gr.Slider(10, 2000, value=256, label="Height", info="Product Height")
                i_width = gr.Slider(10, 2000, value=256, label="Width", info="Product Width")
                i_left = gr.Slider(0, 3000, value=100, label="Towards Right", info="Ref top left corner")
                i_down = gr.Slider(0, 3000, value=100, label="Towards Down", info="Ref top left corner")
                i_btn = gr.Button("Insert Image", variant="primary")
            with gr.Column():
                i_output_image = gr.Image(label="Result Image", type="pil")
                ii_btn = gr.Button("Insert Another Image", variant="secondary")
                    
        i_components = [i_bkg_image, i_prd_image, i_output_image]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(i_components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        i_btn.click(fn=insert_image, inputs=[i_bkg_image, i_prd_image, i_angle, i_height, i_width, i_left, i_down], outputs=i_output_image)
        ii_btn.click(fn=insert_more_images, inputs=i_output_image, outputs=[i_bkg_image, i_prd_image, i_output_image])
    
    with gr.Tab("Insert Logo"):
        with gr.Row():
            with gr.Column():
                l_bkg_image = gr.Image(label="Background Image", type="pil")
            with gr.Column():
                l_prd_image = gr.Image(label="Logo Image", type="pil")
        with gr.Row():
            with gr.Column():
                l_factor = gr.Slider(0.1, 5, value=1, label="Factor", info="Scaling Factor")
                l_opacity = gr.Slider(0, 100, value=80, label="Opacity (%)", info="Opacity")
                l_left = gr.Slider(0, 3000, value=100, label="Towards Right", info="Ref top left corner")
                l_down = gr.Slider(0, 3000, value=100, label="Towards Down", info="Ref top left corner")
                l_btn = gr.Button("Insert Logo", variant="primary")
            with gr.Column():
                l_output_image = gr.Image(label="Result Image", type="pil")
                    
        l_components = [l_bkg_image, l_prd_image, l_output_image]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(l_components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        l_btn.click(fn=AddLogo, inputs=[l_bkg_image, l_prd_image, l_factor, l_opacity, l_left, l_down], outputs=l_output_image)
    
    with gr.Tab("Insert Text"):
        with gr.Row():
            with gr.Column():
                t_bkg_image = gr.Image(label="Background Image", type="pil")
                    
            with gr.Column():
                t_text = gr.Textbox(label="Text", info="Enter Sample Text", value="SAMPLE TEXT")
                t_font = gr.Dropdown(available_fonts, label="Font", info="Select the font", value="Arial", allow_custom_value=True)
                t_size = gr.Slider(10, 200, value=36, label="Font Size", info="Select Font Size")
                t_R = gr.Slider(0, 255, value=255, label="R Component", info="R Color Component Intensity")
                t_G = gr.Slider(0, 255, value=255, label="G Component", info="G Color Component Intensity")
                t_B = gr.Slider(0, 255, value=255, label="B Component", info="B Color Component Intensity")
                t_left = gr.Slider(0, 3000, value=100, label="Towards Right", info="Ref top left corner")
                t_down = gr.Slider(0, 3000, value=100, label="Towards Down", info="Ref top left corner")
                t_btn = gr.Button("Insert Text", variant="primary")
                    
            with gr.Column():
                t_output_image = gr.Image(label="Result Image", type="pil")
                tt_btn = gr.Button("Insert More Text", variant="secondary")
                    
        t_components = [t_bkg_image, t_text, t_output_image]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(t_components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        t_btn.click(fn=AddText, inputs=[t_bkg_image, t_text, t_font, t_size, t_R, t_G, t_B, t_left, t_down], outputs=t_output_image)
        tt_btn.click(fn=AddMoreText, inputs=t_output_image, outputs=[t_bkg_image, t_output_image, t_text])

# Launch configuration for local environment on port 8080
port = int(os.environ.get("PORT", 8080))
demo.queue().launch(server_name="0.0.0.0", server_port=port, share=False)


Running on local URL:  http://127.0.0.1:7861
IMPORTANT: You are using gradio version 4.29.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://64b5214d4ce6108be7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
